In [ ]:
import os
import logging
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from datasets import CubeObstacle, CylinderObstacle, BlockageDataset
from utils.config import Hyperparameters as hp
from utils.tools import calc_sig_strength_gpu

logging.basicConfig(level=logging.WARNING)

def createDirectory(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

def save_df(path: str, name: str, data: list):
    if os.path.exists(f"{path}/{name}"):
        df = pd.read_csv(f"{path}/{name}")
        df = pd.concat([df, pd.DataFrame(data, columns=["gnd1_x", "gnd1_y", "gnd1_z", 
                                                        "gnd2_x", "gnd2_y", "gnd2_z",
                                                        "gnd3_x", "gnd3_y", "gnd3_z", 
                                                        "gnd4_x", "gnd4_y", "gnd4_z",
                                                        "result_x", "result_y", "result_z"])])
    else:
        df = pd.DataFrame(data, columns=["gnd1_x", "gnd1_y", "gnd1_z", 
                                         "gnd2_x", "gnd2_y", "gnd2_z",
                                         "gnd3_x", "gnd3_y", "gnd3_z", 
                                         "gnd4_x", "gnd4_y", "gnd4_z",
                                         "result_x", "result_y", "result_z"])
    createDirectory(path)
    df.to_csv(f"{path}/{name}", index=False)

if __name__ == "__main__":
    result_ls = []
    
    obstacle_ls = [
        CubeObstacle(-30, 25, 35, 60, 20, 0.1),
        CubeObstacle(-30, -25, 45, 10, 35, 0.1),
        CubeObstacle(-30, -60, 35, 60, 20, 0.1),
        CubeObstacle(50, -20, 35, 25, 25, 0.1),
        CylinderObstacle(10, -5,  70, 15, 0.1),
    ]
    grid_step = 0.1
    
    test_height_ls = [50, 60, 70, 80, 90, 100]
    for test_height in test_height_ls:
        print(f"Test Height: {test_height}")
        dataset = BlockageDataset(20000, obstacle_ls, 4, dtype=torch.float32, grid_step=grid_step, height=test_height).to(hp.device)
        df = pd.read_csv('data/gn_coords_4.csv', header=None)
        dataset.gnd_nodes = torch.tensor(df.values, dtype=torch.float32, device=hp.device).reshape(-1, 4, 3)
        logging.info(f"len(dataset): {len(dataset)}")
        
        dataloader = DataLoader(dataset, batch_size=1)
        
        try:
            for i, data in enumerate(tqdm(dataloader)):
                station_pos, gnd_nodes, obst_points = data
                station_pos = station_pos.squeeze(0)
                gnd_nodes = gnd_nodes.squeeze(0)
                obst_points = obst_points.squeeze(0)

                chunk_size = 100000
                sig_chunks = []
                for j in range(0, station_pos.shape[0], chunk_size):
                    station_chunk = station_pos[j:j+chunk_size]  # [chunk_size, 3]
                    sig_chunk = calc_sig_strength_gpu(station_chunk, gnd_nodes, obst_points)
                    sig_chunks.append(sig_chunk)
                sig = torch.cat(sig_chunks, dim=0)
                sig = sig.reshape(dataset.grid_shape[0], dataset.grid_shape[1])

                max_idx = torch.unravel_index(torch.argmax(sig), sig.shape)
                np_gnd_nodes = gnd_nodes.cpu().numpy()
                np_max_pos = (max_idx[0].cpu().numpy()*grid_step, max_idx[1].cpu().numpy()*grid_step, test_height)
                logging.info(f"Max Signal: {sig[max_idx]}, Index: {max_idx}")

                result_ls.append([np_gnd_nodes[0, 0], np_gnd_nodes[0, 1], np_gnd_nodes[0, 2],
                                np_gnd_nodes[1, 0], np_gnd_nodes[1, 1], np_gnd_nodes[1, 2],
                                np_gnd_nodes[2, 0], np_gnd_nodes[2, 1], np_gnd_nodes[2, 2],
                                np_gnd_nodes[3, 0], np_gnd_nodes[3, 1], np_gnd_nodes[3, 2],
                                np_max_pos[0]-(hp.area_size//2), np_max_pos[1]-(hp.area_size//2), np_max_pos[2]])
                logging.info(f"Result: {max_idx}, sig_max: {sig[max_idx]}")

        except KeyboardInterrupt as e:
            logging.warning("Interrupted by user: " + str(e))
        finally:
            save_df("data/result_plots/height", f"height{test_height}_data.csv", result_ls)


In [ ]:
import os
import logging
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from datasets import CubeObstacle, CylinderObstacle, BlockageDataset
from utils.config import Hyperparameters as hp
from utils.tools import calc_sig_strength_gpu

logging.basicConfig(level=logging.WARNING)

def createDirectory(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

def save_df(path: str, name: str, data: list, test_num: int):
    # ground 노드 열 이름 동적 생성
    columns = []
    for i in range(test_num):
        columns.extend([f"gnd{i+1}_x", f"gnd{i+1}_y", f"gnd{i+1}_z"])
    columns.extend(["result_x", "result_y", "result_z"])
    
    # 기존 파일이 있으면 불러와서 이어붙임, 없으면 새 DataFrame 생성
    full_path = os.path.join(path, name)
    if os.path.exists(full_path):
        df = pd.read_csv(full_path)
        new_df = pd.DataFrame(data, columns=columns)
        df = pd.concat([df, new_df], ignore_index=True)
    else:
        df = pd.DataFrame(data, columns=columns)
    
    createDirectory(path)
    df.to_csv(full_path, index=False)

if __name__ == "__main__":
    test_height = 70
    
    obstacle_ls = [
        CubeObstacle(-30, 25, 35, 60, 20, 0.1),
        CubeObstacle(-30, -25, 45, 10, 35, 0.1),
        CubeObstacle(-30, -60, 35, 60, 20, 0.1),
        CubeObstacle(50, -20, 35, 25, 25, 0.1),
        CylinderObstacle(10, -5,  70, 15, 0.1),
    ]
    grid_step = 0.1
    
    test_num_ls = [2, 3, 4, 5, 6, 7, 8]
    for test_num in test_num_ls:
        # 각 test_num 반복마다 결과 리스트 초기화
        result_ls = []
        print(f"Test num of GUs: {test_num}")
        dataset = BlockageDataset(20000, obstacle_ls, test_num, dtype=torch.float32, grid_step=grid_step, height=test_height).to(hp.device)
        df = pd.read_csv(f'data/gn_coords_{test_num}.csv', header=None)
        dataset.gnd_nodes = torch.tensor(df.values, dtype=torch.float32, device=hp.device).reshape(-1, test_num, 3)
        logging.info(f"len(dataset): {len(dataset)}")
        
        dataloader = DataLoader(dataset, batch_size=1)
        
        try:
            for i, data in enumerate(tqdm(dataloader)):
                station_pos, gnd_nodes, obst_points = data
                station_pos = station_pos.squeeze(0)      # [num_candidates, 3]
                gnd_nodes = gnd_nodes.squeeze(0)          # [test_num, 3]
                obst_points = obst_points.squeeze(0)      # [N_c, 3]

                chunk_size = 100000
                sig_chunks = []
                for j in range(0, station_pos.shape[0], chunk_size):
                    station_chunk = station_pos[j:j+chunk_size]  # [chunk_size, 3]
                    sig_chunk = calc_sig_strength_gpu(station_chunk, gnd_nodes, obst_points)
                    sig_chunks.append(sig_chunk)
                sig = torch.cat(sig_chunks, dim=0)
                sig = sig.reshape(dataset.grid_shape[0], dataset.grid_shape[1])

                max_idx = torch.unravel_index(torch.argmax(sig), sig.shape)
                np_gnd_nodes = gnd_nodes.cpu().numpy()        # shape: [test_num, 3]
                flat_gnd = np_gnd_nodes.flatten()             # 1D 배열, 길이 = test_num*3

                np_max_pos = (max_idx[0].cpu().numpy()*grid_step - (hp.area_size//2),
                              max_idx[1].cpu().numpy()*grid_step - (hp.area_size//2),
                              test_height)
                logging.info(f"Max Signal: {sig[max_idx]}, Index: {max_idx}")

                # flat_gnd의 리스트와 np_max_pos를 합쳐서 결과 저장
                result_ls.append(list(flat_gnd) + [np_max_pos[0], np_max_pos[1], np_max_pos[2]])
                logging.info(f"Result: {max_idx}, sig_max: {sig[max_idx]}")

        except KeyboardInterrupt as e:
            logging.warning("Interrupted by user: " + str(e))
        finally:
            save_df("data/result_plots/num_of_GUs", f"num{test_num}_data.csv", result_ls, test_num)
